In [1]:
%pip install "flaml[automl]" pandas numpy scikit-learn matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


In [4]:
date_candidates = [
    c for c in data.columns
    if c in ["date", "datetime", "timestamp"]
]

if date_candidates:
    date_col = date_candidates[0]

    data[date_col] = pd.to_datetime(
        data[date_col],
        errors="coerce",
        utc=True
    )

    data = data.dropna(subset=[date_col])
    data = data.sort_values(date_col).reset_index(drop=True)
else:
    print("Warning: No date column found. "
          "Rows will be assumed to be chronological.")

In [6]:
import pandas as pd
import numpy as np

# Change this only if your CSV has a different filename
FILE_PATH = "apple_historical_data.csv"

data = pd.read_csv(FILE_PATH)

# Standardize column names
data.columns = (
    data.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Recognize common column-name variations
aliases = {
    "adj_close": "close",
    "daily_return": "daily_return",
    "ma_20": "ma20",
    "ma_50": "ma50",
    "stock_splits": "stock_splits"
}

data = data.rename(columns=aliases)

# Find the date column, if available
date_candidates = [
    c for c in data.columns
    if c in ["date", "datetime", "timestamp"]
]

if date_candidates:
    data[date_col] = pd.to_datetime(
    data[date_col],
    errors="coerce",
    utc=True
)
    data = data.dropna(subset=[date_col])
    data = data.sort_values(date_col).reset_index(drop=True)
else:
    print("Warning: No date column found. "
          "Rows will be assumed to be chronological.")

if "close" not in data.columns:
    raise ValueError(
        f"Closing-price column not found. "
        f"Available columns: {data.columns.tolist()}"
    )

data["close"] = pd.to_numeric(data["close"], errors="coerce")
data = data.dropna(subset=["close"]).copy()

# Recalculate the features consistently
data["daily_return"] = data["close"].pct_change()
data["ma20"] = data["close"].rolling(20).mean()
data["ma50"] = data["close"].rolling(50).mean()
data["volatility"] = (
    data["daily_return"].rolling(20).std()
)
data["momentum"] = data["close"].pct_change(5)

# Additional financial features
data["ma20_ratio"] = data["close"] / data["ma20"] - 1
data["ma50_ratio"] = data["close"] / data["ma50"] - 1

# RSI calculated from past price changes
delta = data["close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)
data["rsi"] = 100 - (100 / (1 + rs))

# Optional volume features
if "volume" in data.columns:
    data["volume_change"] = data["volume"].pct_change()
    data["volume_ma20_ratio"] = (
        data["volume"] /
        data["volume"].rolling(20).mean() - 1
    )

# Next-day closing-price direction
data["next_close"] = data["close"].shift(-1)
data["next_return"] = data["next_close"] / data["close"] - 1
data["target"] = (data["next_return"] > 0).astype(int)

print("Dataset shape:", data.shape)
print(data.columns.tolist())
print(data.tail())

Dataset shape: (502, 22)
['unnamed:_0', 'date', 'open', 'high', 'low', 'close', 'volume', 'dividends', 'stock_splits', 'ma20', 'ma50', 'daily_return', 'volatility', 'momentum', 'ma20_ratio', 'ma50_ratio', 'rsi', 'volume_change', 'volume_ma20_ratio', 'next_close', 'next_return', 'target']
     unnamed:_0                      date        open        high         low  \
497         497 2026-10-09 04:00:00+00:00  331.700012  338.609985  330.700012   
498         498 2026-10-12 04:00:00+00:00  338.220001  342.630005  337.600006   
499         499 2026-10-13 04:00:00+00:00  344.670013  348.149994  340.739990   
500         500 2026-10-14 04:00:00+00:00  344.609985  347.089996  341.720001   
501         501 2026-10-15 04:00:00+00:00  346.950012  347.850006  344.250000   

          close    volume  dividends  stock_splits        ma20  ...  \
497  336.640015  37881200        0.0           0.0  335.389003  ...   
498  340.970001  25900600        0.0           0.0  335.783504  ...   
499  341.26

In [1]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Remove rows where indicators are not yet available
data = data.replace([np.inf, -np.inf], np.nan)

feature_cols = [
    "daily_return", "ma20_ratio", "ma50_ratio",
    "volatility", "momentum", "rsi"
]

if "volume_change" in data.columns:
    feature_cols.append("volume_change")
if "volume_ma20_ratio" in data.columns:
    feature_cols.append("volume_ma20_ratio")

data = data.dropna(
    subset=feature_cols + ["target", "next_return"]
).copy()

# Chronological split: first 80% for development, last 20% for testing
split = int(len(data) * 0.80)

if split < 100 or len(data) - split < 20:
    raise ValueError(
        "Not enough usable data for this configuration. "
        "Check your dataset size."
    )

regime_cols = ["daily_return", "volatility"]

regime_scaler = StandardScaler()
regime_scaler.fit(data.iloc[:split][regime_cols])

train_regime_x = regime_scaler.transform(
    data.iloc[:split][regime_cols]
)
all_regime_x = regime_scaler.transform(data[regime_cols])

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

kmeans.fit(train_regime_x)

data["regime_cluster"] = kmeans.predict(all_regime_x)

# Name the clusters by their average training-period return
cluster_returns = data.iloc[:split].groupby(
    "regime_cluster"
)["daily_return"].mean().sort_values()

ordered_clusters = cluster_returns.index.tolist()

regime_names = {
    ordered_clusters[0]: "Bearish",
    ordered_clusters[1]: "Sideways",
    ordered_clusters[2]: "Bullish"
}

data["market_regime"] = (
    data["regime_cluster"].map(regime_names)
)

print(data["market_regime"].value_counts())
print(data[[
    "daily_return", "volatility", "market_regime"
]].tail())

NameError: name 'data' is not defined

In [ ]:
# Encode market regimes as separate binary features
regime_dummies = pd.get_dummies(
    data["market_regime"],
    prefix="regime",
    dtype=int
)

data = pd.concat([data, regime_dummies], axis=1)

regime_feature_cols = list(regime_dummies.columns)

# Only information available on the prediction date
feature_cols = [
    "daily_return",
    "ma20_ratio",
    "ma50_ratio",
    "volatility",
    "momentum",
    "rsi"
]

for col in ["volume_change", "volume_ma20_ratio"]:
    if col in data.columns:
        feature_cols.append(col)

feature_cols += regime_feature_cols

X = data[feature_cols].copy()
y = data["target"].astype(int)

# Retain the chronological test period
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", feature_cols)
print("Training target distribution:")
print(y_train.value_counts())

Training rows: 361
Testing rows: 91
Features: ['daily_return', 'ma20_ratio', 'ma50_ratio', 'volatility', 'momentum', 'rsi', 'volume_change', 'volume_ma20_ratio', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways']
Training target distribution:
target
1    188
0    173
Name: count, dtype: int64


In [ ]:
from flaml import AutoML

print("FLAML imported successfully!")

FLAML imported successfully!


In [ ]:
from flaml import AutoML
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import joblib

automl = AutoML()

automl.fit(
    X_train=X_train,
    y_train=y_train,

    task="classification",
    metric="log_loss",

    # Limit the search so it finishes in reasonable time
    time_budget=120,

    # Preserve the order of financial observations
    eval_method="holdout",
    split_type="time",
    split_ratio=0.20,

    # Models FLAML can search among
    estimator_list=[
        "lrl1",
        "lrl2",
        "rf",
        "extra_tree",
        "xgboost",
        "lgbm"
    ],

    seed=42,
    n_jobs=-1,
    verbose=1
)

print("Best model:", automl.best_estimator)
print("Best validation loss:", automl.best_loss)
print("Best model configuration:", automl.best_config)

# Save the trained AutoML object
joblib.dump(automl, "automl_stock_model.pkl")

Best model: lgbm
Best validation loss: 0.6456864265806752
Best model configuration: {'n_estimators': 287, 'num_leaves': 11, 'min_child_samples': 97, 'learning_rate': np.float64(0.46408995963774713), 'log_max_bin': 8, 'colsample_bytree': np.float64(0.5605588136740927), 'reg_alpha': np.float64(0.5241062325560008), 'reg_lambda': np.float64(0.16342251903181987)}


['automl_stock_model.pkl']

In [ ]:
# Predict the test set
y_pred = automl.predict(X_test)
y_proba = automl.predict_proba(X_test)

# Find the probability column corresponding to class 1 (UP)
class_list = list(automl.classes_)
up_index = class_list.index(1)
p_up = y_proba[:, up_index]

print("Accuracy:", round(
    accuracy_score(y_test, y_pred), 4
))

print("Balanced accuracy:", round(
    balanced_accuracy_score(y_test, y_pred), 4
))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred, zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

if y_test.nunique() == 2:
    print("ROC-AUC:", round(
        roc_auc_score(y_test, p_up), 4
    ))
else:
    print("ROC-AUC unavailable: test set has one class.")

Accuracy: 0.5604
Balanced accuracy: 0.5647

Classification report:
              precision    recall  f1-score   support

           0       0.50      0.60      0.55        40
           1       0.63      0.53      0.57        51

    accuracy                           0.56        91
   macro avg       0.56      0.56      0.56        91
weighted avg       0.57      0.56      0.56        91

Confusion matrix:
[[24 16]
 [24 27]]
ROC-AUC: 0.5696


In [ ]:
# Build a test-results table
test_results = data.iloc[split:].copy()

test_results["predicted_direction"] = y_pred
test_results["probability_up"] = p_up

# Estimate the high-volatility threshold from training data only
volatility_limit = data.iloc[:split][
    "volatility"
].quantile(0.75)

def make_signal(row):
    probability = row["probability_up"]
    volatility = row["volatility"]

    if probability >= 0.60 and volatility <= volatility_limit:
        return "BUY"
    elif probability <= 0.40:
        return "AVOID"
    else:
        return "HOLD"

test_results["signal"] = test_results.apply(
    make_signal, axis=1
)

print(test_results[[
    "close", "market_regime",
    "probability_up", "volatility", "signal"
]].tail(15))

          close market_regime  probability_up  volatility signal
486  335.920013       Bullish        0.672385    0.013820    BUY
487  341.070007       Bullish        0.450704    0.014069   HOLD
488  338.399994       Bearish        0.366361    0.014007  AVOID
489  329.399994       Bearish        0.464438    0.015294   HOLD
490  333.019989       Bullish        0.905877    0.014387    BUY
491  330.320007       Bearish        0.360354    0.014536  AVOID
492  333.690002       Bullish        0.707918    0.014543    BUY
493  332.890015       Bullish        0.656371    0.013231    BUY
494  333.630005       Bullish        0.616806    0.012828    BUY
495  336.670013       Bullish        0.519537    0.012833   HOLD
496  340.420013       Bullish        0.282612    0.010561  AVOID
497  336.640015       Bearish        0.640239    0.010308    BUY
498  340.970001       Bullish        0.750459    0.010657    BUY
499  341.269989       Bullish        0.466810    0.010550   HOLD
500  342.959991       Bul

In [ ]:
# Long-only strategy: BUY = 1, otherwise 0
test_results["position"] = (
    test_results["signal"] == "BUY"
).astype(int)

# next_return is the return from today's close to tomorrow's close
test_results["buy_hold_return"] = test_results["next_return"]

# Charge 0.1% per position change (illustrative assumption)
transaction_cost = 0.001

test_results["turnover"] = (
    test_results["position"].diff().abs()
)
test_results.loc[
    test_results.index[0], "turnover"
] = test_results["position"].iloc[0]

test_results["strategy_return"] = (
    test_results["position"] * test_results["next_return"]
    - transaction_cost * test_results["turnover"]
)

def performance_metrics(returns):
    returns = returns.dropna()

    if len(returns) == 0:
        return {}

    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1

    std = returns.std()
    downside = returns[returns < 0]

    downside_std = (
        np.sqrt((downside ** 2).mean())
        if len(downside) else np.nan
    )

    sharpe = (
        np.sqrt(252) * returns.mean() / std
        if std > 0 else np.nan
    )

    sortino = (
        np.sqrt(252) * returns.mean() / downside_std
        if pd.notna(downside_std) and downside_std > 0
        else np.nan
    )

    return {
        "Total return (%)": (equity.iloc[-1] - 1) * 100,
        "Annualized volatility (%)": std * np.sqrt(252) * 100,
        "Sharpe ratio": sharpe,
        "Sortino ratio": sortino,
        "Maximum drawdown (%)": drawdown.min() * 100
    }

strategy_metrics = performance_metrics(
    test_results["strategy_return"]
)

buy_hold_metrics = performance_metrics(
    test_results["buy_hold_return"]
)

comparison = pd.DataFrame({
    "AutoML Strategy": strategy_metrics,
    "Buy & Hold": buy_hold_metrics
})

print(comparison.round(3))

# Create equity curves for the later dashboard
test_results["strategy_equity"] = (
    1 + test_results["strategy_return"]
).cumprod()

test_results["buy_hold_equity"] = (
    1 + test_results["buy_hold_return"]
).cumprod()

test_results.to_csv("automl_backtest_results.csv", index=False)
comparison.to_csv("automl_performance_comparison.csv")

print("\nBacktest results saved.")

                           AutoML Strategy  Buy & Hold
Total return (%)                    -2.102      12.205
Annualized volatility (%)           13.400      28.582
Sharpe ratio                        -0.371       1.259
Sortino ratio                       -0.219       1.157
Maximum drawdown (%)                -8.678     -11.047

Backtest results saved.


In [ ]:
# Save the cleaned dataset and all key outputs
data.to_csv("apple_features_and_regimes.csv", index=False)

joblib.dump(
    {
        "regime_scaler": regime_scaler,
        "kmeans": kmeans,
        "regime_names": regime_names,
        "feature_columns": feature_cols,
        "volatility_limit": volatility_limit
    },
    "market_regime_artifacts.pkl"
)

print("PROJECT OUTPUTS SAVED")
print("- apple_features_and_regimes.csv")
print("- automl_stock_model.pkl")
print("- market_regime_artifacts.pkl")
print("- automl_backtest_results.csv")
print("- automl_performance_comparison.csv")

PROJECT OUTPUTS SAVED
- apple_features_and_regimes.csv
- automl_stock_model.pkl
- market_regime_artifacts.pkl
- automl_backtest_results.csv
- automl_performance_comparison.csv


In [ ]:
import pandas as pd
import os

files = [
    "apple_features_and_regimes.csv",
    "automl_backtest_results.csv",
    "automl_performance_comparison.csv"
]

for file in files:
    if os.path.exists(file):
        print(f"\n{'=' * 50}")
        print(file)
        print(pd.read_csv(file).head())
    else:
        print(f"\nNot found: {file}")


apple_features_and_regimes.csv
   unnamed:_0                       date        open        high         low  \
0          49  2024-12-24 05:00:00+00:00  253.649270  256.349660  253.450699   
1          50  2024-12-26 05:00:00+00:00  256.329832  258.226075  255.773869   
2          51  2024-12-27 05:00:00+00:00  255.972387  256.836144  251.236764   
3          52  2024-12-30 05:00:00+00:00  250.412763  251.673617  248.943430   
4          53  2024-12-31 05:00:00+00:00  250.621250  251.455194  247.632926   

        close    volume  dividends  stock_splits        ma20  ...  \
0  256.339752  23234700        0.0           0.0  244.519022  ...   
1  257.153839  27237100        0.0           0.0  245.708392  ...   
2  253.748535  42355300        0.0           0.0  246.733949  ...   
3  250.382980  35557500        0.0           0.0  247.472092  ...   
4  248.615799  39480700        0.0           0.0  248.009692  ...   

   volume_change  volume_ma20_ratio  next_close  next_return  target  \


In [ ]:
import pandas as pd
import os

files = [
    "automl_performance_comparison.csv",
    "automl_backtest_results.csv"
]

for file in files:
    print("\n" + "=" * 60)
    print(file)

    if os.path.exists(file):
        df = pd.read_csv(file)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.to_string(index=False))
    else:
        print("File not found.")


automl_performance_comparison.csv
Shape: (5, 3)
Columns: ['Unnamed: 0', 'AutoML Strategy', 'Buy & Hold']
               Unnamed: 0  AutoML Strategy  Buy & Hold
         Total return (%)        -2.101682   12.205425
Annualized volatility (%)        13.399750   28.581644
             Sharpe ratio        -0.371195    1.259297
            Sortino ratio        -0.219042    1.156742
     Maximum drawdown (%)        -8.677706  -11.047196

automl_backtest_results.csv
Shape: (91, 36)
Columns: ['unnamed:_0', 'date', 'open', 'high', 'low', 'close', 'volume', 'dividends', 'stock_splits', 'ma20', 'ma50', 'daily_return', 'volatility', 'momentum', 'ma20_ratio', 'ma50_ratio', 'rsi', 'volume_change', 'volume_ma20_ratio', 'next_close', 'next_return', 'target', 'regime_cluster', 'market_regime', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways', 'predicted_direction', 'probability_up', 'signal', 'position', 'buy_hold_return', 'turnover', 'strategy_return', 'strategy_equity', 'buy_hold_equity']
 unna

In [ ]:
# TEST 1: Check for future-information leakage

print("Training feature columns:")
print(list(X_train.columns))

print("\nTesting feature columns:")
print(list(X_test.columns))

forbidden_columns = {
    "target",
    "next_close",
    "next_return",
    "predicted_direction",
    "probability_up",
    "signal",
    "position",
    "strategy_return",
    "strategy_equity",
    "buy_hold_return",
    "buy_hold_equity"
}

leaked_train = forbidden_columns.intersection(X_train.columns)
leaked_test = forbidden_columns.intersection(X_test.columns)

print("\nForbidden columns in X_train:", leaked_train)
print("Forbidden columns in X_test:", leaked_test)

if not leaked_train and not leaked_test:
    print("\nPASS: No obvious target/outcome columns found in model inputs.")
else:
    print("\nWARNING: Potential leakage. Investigate before proceeding.")

Training feature columns:
['daily_return', 'ma20_ratio', 'ma50_ratio', 'volatility', 'momentum', 'rsi', 'volume_change', 'volume_ma20_ratio', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways']

Testing feature columns:
['daily_return', 'ma20_ratio', 'ma50_ratio', 'volatility', 'momentum', 'rsi', 'volume_change', 'volume_ma20_ratio', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways']

Forbidden columns in X_train: set()
Forbidden columns in X_test: set()

PASS: No obvious target/outcome columns found in model inputs.


In [ ]:
# TEST 2: Check chronological separation

train_dates = pd.to_datetime(
    data.loc[X_train.index, "date"],
    utc=True
)

test_dates = pd.to_datetime(
    data.loc[X_test.index, "date"],
    utc=True
)

print("Last training date:", train_dates.max())
print("First testing date:", test_dates.min())

print(
    "Training dates precede testing dates:",
    train_dates.max() < test_dates.min()
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Last training date: 2026-06-04 04:00:00+00:00
First testing date: 2026-06-05 04:00:00+00:00
Training dates precede testing dates: True
Training rows: 361
Testing rows: 91


In [ ]:
backtest = pd.read_csv("automl_backtest_results.csv")

In [ ]:
# Inspect the saved backtest columns
print(
    backtest[
        [
            "date",
            "signal",
            "position",
            "turnover",
            "next_return",
            "strategy_return"
        ]
    ].head(15).to_string(index=False)
)

                     date signal  position  turnover  next_return  strategy_return
2026-06-05 04:00:00+00:00  AVOID         0       0.0    -0.018872        -0.000000
2026-06-08 04:00:00+00:00  AVOID         0       0.0    -0.036446        -0.000000
2026-06-09 04:00:00+00:00    BUY         1       1.0     0.003545         0.002545
2026-06-10 04:00:00+00:00   HOLD         0       1.0     0.013890        -0.001000
2026-06-11 04:00:00+00:00    BUY         1       1.0    -0.015222        -0.016222
2026-06-12 04:00:00+00:00    BUY         1       0.0     0.018171         0.018171
2026-06-15 04:00:00+00:00  AVOID         0       1.0     0.009514        -0.001000
2026-06-16 04:00:00+00:00   HOLD         0       0.0    -0.010995        -0.000000
2026-06-17 04:00:00+00:00    BUY         1       1.0     0.006961         0.005961
2026-06-18 04:00:00+00:00    BUY         1       0.0    -0.003356        -0.003356
2026-06-22 04:00:00+00:00    BUY         1       0.0    -0.009124        -0.009124
2026

In [ ]:
print("First and last strategy equity:")
print(
    backtest[
        ["date", "strategy_equity", "buy_hold_equity"]
    ].iloc[[0, -1]].to_string(index=False)
)

print("\nPerformance summary:")
print(pd.read_csv("automl_performance_comparison.csv").to_string(index=False))

First and last strategy equity:
                     date  strategy_equity  buy_hold_equity
2026-06-05 04:00:00+00:00         1.000000         0.981128
2026-10-14 04:00:00+00:00         0.978983         1.122054

Performance summary:
               Unnamed: 0  AutoML Strategy  Buy & Hold
         Total return (%)        -2.101682   12.205425
Annualized volatility (%)        13.399750   28.581644
             Sharpe ratio        -0.371195    1.259297
            Sortino ratio        -0.219042    1.156742
     Maximum drawdown (%)        -8.677706  -11.047196


In [ ]:
print("First 5 rows:")
print(
    backtest[
        ["date", "next_return", "buy_hold_return",
         "strategy_return", "strategy_equity", "buy_hold_equity"]
    ].head().to_string(index=False)
)

print("\nEquity calculation check:")

strategy_equity_check = (
    1 + backtest["strategy_return"]
).cumprod()

buy_hold_equity_check = (
    1 + backtest["buy_hold_return"]
).cumprod()

print("Stored strategy final equity:",
      backtest["strategy_equity"].iloc[-1])

print("Recalculated strategy final equity:",
      strategy_equity_check.iloc[-1])

print("Stored Buy & Hold final equity:",
      backtest["buy_hold_equity"].iloc[-1])

print("Recalculated Buy & Hold final equity:",
      buy_hold_equity_check.iloc[-1])

First 5 rows:
                     date  next_return  buy_hold_return  strategy_return  strategy_equity  buy_hold_equity
2026-06-05 04:00:00+00:00    -0.018872        -0.018872        -0.000000         1.000000         0.981128
2026-06-08 04:00:00+00:00    -0.036446        -0.036446        -0.000000         1.000000         0.945370
2026-06-09 04:00:00+00:00     0.003545         0.003545         0.002545         1.002545         0.948721
2026-06-10 04:00:00+00:00     0.013890         0.013890        -0.001000         1.001542         0.961899
2026-06-11 04:00:00+00:00    -0.015222        -0.015222        -0.016222         0.985296         0.947257

Equity calculation check:
Stored strategy final equity: 0.978983179238377
Recalculated strategy final equity: 0.978983179238377
Stored Buy & Hold final equity: 1.1220542465747445
Recalculated Buy & Hold final equity: 1.1220542465747467


In [ ]:
print("Search results for backtesting calculations:")

for i, cell in enumerate(get_ipython().user_ns.get("In", [])):
    if any(term in cell for term in [
        "buy_hold_equity",
        "strategy_equity",
        "strategy_return",
        "turnover"
    ]):
        print(f"\n- Cell In[{i}] ->")
        print(cell[:2500])

Search results for backtesting calculations:

- Cell In[16] ->
# Long-only strategy: BUY = 1, otherwise 0
test_results["position"] = (
    test_results["signal"] == "BUY"
).astype(int)

# next_return is the return from today's close to tomorrow's close
test_results["buy_hold_return"] = test_results["next_return"]

# Charge 0.1% per position change (illustrative assumption)
transaction_cost = 0.001

test_results["turnover"] = (
    test_results["position"].diff().abs()
)
test_results.loc[
    test_results.index[0], "turnover"
] = test_results["position"].iloc[0]

test_results["strategy_return"] = (
    test_results["position"] * test_results["next_return"]
    - transaction_cost * test_results["turnover"]
)

def performance_metrics(returns):
    returns = returns.dropna()

    if len(returns) == 0:
        return {}

    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1

    std = returns.std()
    downside = returns[returns < 0]

    downside_std = (
        

In [ ]:
print(In[16])

# Long-only strategy: BUY = 1, otherwise 0
test_results["position"] = (
    test_results["signal"] == "BUY"
).astype(int)

# next_return is the return from today's close to tomorrow's close
test_results["buy_hold_return"] = test_results["next_return"]

# Charge 0.1% per position change (illustrative assumption)
transaction_cost = 0.001

test_results["turnover"] = (
    test_results["position"].diff().abs()
)
test_results.loc[
    test_results.index[0], "turnover"
] = test_results["position"].iloc[0]

test_results["strategy_return"] = (
    test_results["position"] * test_results["next_return"]
    - transaction_cost * test_results["turnover"]
)

def performance_metrics(returns):
    returns = returns.dropna()

    if len(returns) == 0:
        return {}

    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1

    std = returns.std()
    downside = returns[returns < 0]

    downside_std = (
        np.sqrt((downside ** 2).mean())
        if len(downside) else n

In [ ]:
print("\n".join(In[16].splitlines()[15:65]))

    test_results.index[0], "turnover"
] = test_results["position"].iloc[0]

test_results["strategy_return"] = (
    test_results["position"] * test_results["next_return"]
    - transaction_cost * test_results["turnover"]
)

def performance_metrics(returns):
    returns = returns.dropna()

    if len(returns) == 0:
        return {}

    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1

    std = returns.std()
    downside = returns[returns < 0]

    downside_std = (
        np.sqrt((downside ** 2).mean())
        if len(downside) else np.nan
    )

    sharpe = (
        np.sqrt(252) * returns.mean() / std
        if std > 0 else np.nan
    )

    sortino = (
        np.sqrt(252) * returns.mean() / downside_std
        if pd.notna(downside_std) and downside_std > 0
        else np.nan
    )

    return {
        "Total return (%)": (equity.iloc[-1] - 1) * 100,
        "Annualized volatility (%)": std * np.sqrt(252) * 100,
        "Sharpe ratio": sharpe,
    

In [ ]:
print("\n".join(
    line for line in In[16].splitlines()
    if any(term in line for term in [
        "strategy_equity",
        "buy_hold_equity",
        "strategy_metrics",
        "buy_hold_metrics",
        "comparison"
    ])
))

strategy_metrics = performance_metrics(
buy_hold_metrics = performance_metrics(
comparison = pd.DataFrame({
    "AutoML Strategy": strategy_metrics,
    "Buy & Hold": buy_hold_metrics
print(comparison.round(3))
test_results["strategy_equity"] = (
test_results["buy_hold_equity"] = (
comparison.to_csv("automl_performance_comparison.csv")


In [ ]:
for i, cell in enumerate(In):
    if "predict_proba" in cell or "predicted_direction" in cell:
        print(f"\n--- Cell In[{i}] ---")
        print(cell[:2000])


--- Cell In[14] ---
# Predict the test set
y_pred = automl.predict(X_test)
y_proba = automl.predict_proba(X_test)

# Find the probability column corresponding to class 1 (UP)
class_list = list(automl.classes_)
up_index = class_list.index(1)
p_up = y_proba[:, up_index]

print("Accuracy:", round(
    accuracy_score(y_test, y_pred), 4
))

print("Balanced accuracy:", round(
    balanced_accuracy_score(y_test, y_pred), 4
))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred, zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

if y_test.nunique() == 2:
    print("ROC-AUC:", round(
        roc_auc_score(y_test, p_up), 4
    ))
else:
    print("ROC-AUC unavailable: test set has one class.")

--- Cell In[15] ---
# Build a test-results table
test_results = data.iloc[split:].copy()

test_results["predicted_direction"] = y_pred
test_results["probability_up"] = p_up

# Estimate the high-volatility threshold from training data only

In [ ]:
print("TEST SET CREATION:")
print(In[13][:3000])

print("\nPREDICTION AND BACKTEST ALIGNMENT:")
print(In[16][:3000])

TEST SET CREATION:
from flaml import AutoML
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import joblib

automl = AutoML()

automl.fit(
    X_train=X_train,
    y_train=y_train,

    task="classification",
    metric="log_loss",

    # Limit the search so it finishes in reasonable time
    time_budget=120,

    # Preserve the order of financial observations
    eval_method="holdout",
    split_type="time",
    split_ratio=0.20,

    # Models FLAML can search among
    estimator_list=[
        "lrl1",
        "lrl2",
        "rf",
        "extra_tree",
        "xgboost",
        "lgbm"
    ],

    seed=42,
    n_jobs=-1,
    verbose=1
)

print("Best model:", automl.best_estimator)
print("Best validation loss:", automl.best_loss)
print("Best model configuration:", automl.best_config)

# Save the trained AutoML object
joblib.dump(automl, "automl_stock_model.pkl")

PREDICTION AND BACKTEST 

In [ ]:
print("\n".join(
    f"Cell In[{i}]: {line.strip()}"
    for i, cell in enumerate(In)
    for line in cell.splitlines()
    if any(term in line for term in [
        "train_test_split",
        "X_train =",
        "X_test =",
        "test_results =",
        "predicted_direction",
        "p_up"
    ])
))

Cell In[8]: X_train = X.iloc[:split]
Cell In[8]: X_test = X.iloc[split:]
Cell In[14]: p_up = y_proba[:, up_index]
Cell In[14]: roc_auc_score(y_test, p_up), 4
Cell In[15]: test_results = data.iloc[split:].copy()
Cell In[15]: test_results["predicted_direction"] = y_pred
Cell In[15]: test_results["probability_up"] = p_up
Cell In[20]: "predicted_direction",
Cell In[38]: if "predict_proba" in cell or "predicted_direction" in cell:
Cell In[40]: "train_test_split",
Cell In[40]: "X_train =",
Cell In[40]: "X_test =",
Cell In[40]: "test_results =",
Cell In[40]: "predicted_direction",
Cell In[40]: "p_up"


In [ ]:
print("X_test rows:", len(X_test))
print("Test result rows:", len(test_results))

print("\nIndexes match:", X_test.index.equals(test_results.index))

print("\nFirst 5 matching dates:")
print(data.loc[X_test.index, "date"].head().to_list())
print(test_results["date"].head().to_list())

print("\nTarget alignment:")
print("Target indexes match:", y_test.index.equals(test_results.index))

X_test rows: 91
Test result rows: 91

Indexes match: True

First 5 matching dates:
[Timestamp('2026-06-05 04:00:00+0000', tz='UTC'), Timestamp('2026-06-08 04:00:00+0000', tz='UTC'), Timestamp('2026-06-09 04:00:00+0000', tz='UTC'), Timestamp('2026-06-10 04:00:00+0000', tz='UTC'), Timestamp('2026-06-11 04:00:00+0000', tz='UTC')]
[Timestamp('2026-06-05 04:00:00+0000', tz='UTC'), Timestamp('2026-06-08 04:00:00+0000', tz='UTC'), Timestamp('2026-06-09 04:00:00+0000', tz='UTC'), Timestamp('2026-06-10 04:00:00+0000', tz='UTC'), Timestamp('2026-06-11 04:00:00+0000', tz='UTC')]

Target alignment:
Target indexes match: True


In [ ]:
from pathlib import Path
import pickle

project_folder = Path.cwd()

model_path = project_folder / "automl_stock_model.pkl"
regime_path = project_folder / "market_regime_artifacts.pkl"

print("Project folder:", project_folder)
print("Model file exists:", model_path.exists())
print("Regime file exists:", regime_path.exists())

if model_path.exists() and regime_path.exists():
    with open(model_path, "rb") as file:
        model = pickle.load(file)

    with open(regime_path, "rb") as file:
        regime_artifacts = pickle.load(file)

    print("\nModel loaded successfully!")
    print("Model type:", type(model).__name__)

    print("\nRegime artifacts loaded successfully!")
    print("Artifact type:", type(regime_artifacts).__name__)

    if isinstance(regime_artifacts, dict):
        print("Artifact keys:", list(regime_artifacts.keys()))
else:
    print("\nFiles not found in the current folder.")

Project folder: /Users/atishi/Desktop/C++ Program/automl-stock-project
Model file exists: True
Regime file exists: True


UnpicklingError: STACK_GLOBAL requires str

In [ ]:
from pathlib import Path

model_path = Path("automl_stock_model.pkl")
regime_path = Path("market_regime_artifacts.pkl")

for path in [model_path, regime_path]:
    print("\nFile:", path.name)
    print("Exists:", path.exists())

    if path.exists():
        print("Size:", path.stat().st_size, "bytes")

        with open(path, "rb") as file:
            print("First 30 bytes:", file.read(30))


File: automl_stock_model.pkl
Exists: True
Size: 2527347 bytes
First 30 bytes: b'\x80\x05\x95}\x0f\x00\x00\x00\x00\x00\x00\x8c\x13flaml.automl.auto'

File: market_regime_artifacts.pkl
Exists: True
Size: 3176 bytes
First 30 bytes: b'\x80\x05\x95P\x01\x00\x00\x00\x00\x00\x00}\x94(\x8c\rregime_scaler\x94'


In [ ]:
import joblib

model = joblib.load("automl_stock_model.pkl")
regime_artifacts = joblib.load("market_regime_artifacts.pkl")

print("Model loaded successfully!")
print("Model type:", type(model).__name__)

print("\nRegime artifacts loaded successfully!")
print("Artifact keys:", list(regime_artifacts.keys()))

Model loaded successfully!
Model type: AutoML

Regime artifacts loaded successfully!
Artifact keys: ['regime_scaler', 'kmeans', 'regime_names', 'feature_columns', 'volatility_limit']


In [ ]:
print("MODEL INFORMATION")
print("-" * 40)

print("Best model:", model.best_estimator)
print("Best model type:", type(model.model).__name__)

print("\nREGIME DETECTION FEATURES")
print("-" * 40)

print("Feature columns:")
print(regime_artifacts["feature_columns"])

print("\nRegime names:")
print(regime_artifacts["regime_names"])

print("\nVolatility limit:")
print(regime_artifacts["volatility_limit"])

print("\nREGIME SCALER")
print("-" * 40)

print("Scaler type:", type(regime_artifacts["regime_scaler"]).__name__)

print("\nKMEANS MODEL")
print("-" * 40)

print("KMeans type:", type(regime_artifacts["kmeans"]).__name__)
print("Number of clusters:", regime_artifacts["kmeans"].n_clusters)

MODEL INFORMATION
----------------------------------------
Best model: lgbm
Best model type: LGBMEstimator

REGIME DETECTION FEATURES
----------------------------------------
Feature columns:
['daily_return', 'ma20_ratio', 'ma50_ratio', 'volatility', 'momentum', 'rsi', 'volume_change', 'volume_ma20_ratio', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways']

Regime names:
{1: 'Bearish', 2: 'Sideways', 0: 'Bullish'}

Volatility limit:
0.018774586515419106

REGIME SCALER
----------------------------------------
Scaler type: StandardScaler

KMEANS MODEL
----------------------------------------
KMeans type: KMeans
Number of clusters: 3


In [ ]:
print("MODEL INPUT FEATURES")
print("-" * 40)

print("Number of features:", len(model.feature_names_in_))
print("Feature names:")

for feature in model.feature_names_in_:
    print("-", feature)

print("\nMODEL DETAILS")
print("-" * 40)

print("Estimator:", model.best_estimator)
print("Task:", model.task)
print("Classes:", model.classes_)

MODEL INPUT FEATURES
----------------------------------------
Number of features: 11
Feature names:
- daily_return
- ma20_ratio
- ma50_ratio
- volatility
- momentum
- rsi
- volume_change
- volume_ma20_ratio
- regime_Bearish
- regime_Bullish
- regime_Sideways

MODEL DETAILS
----------------------------------------
Estimator: lgbm


AttributeError: 'AutoML' object has no attribute 'task'

In [ ]:
print("FEATURE ENGINEERING CODE")
print("-" * 40)

print("Find the cells in your notebook where these columns are created:")
print("1. daily_return")
print("2. ma20_ratio")
print("3. ma50_ratio")
print("4. volatility")
print("5. momentum")
print("6. rsi")
print("7. volume_change")
print("8. volume_ma20_ratio")

FEATURE ENGINEERING CODE
----------------------------------------
Find the cells in your notebook where these columns are created:
1. daily_return
2. ma20_ratio
3. ma50_ratio
4. volatility
5. momentum
6. rsi
7. volume_change
8. volume_ma20_ratio


In [ ]:
print("Saved regime detector input features:")
print(regime_artifacts["feature_columns"])

print("\nSaved scaler expects this many features:")
print(regime_artifacts["regime_scaler"].n_features_in_)

print("\nSaved KMeans cluster centers shape:")
print(regime_artifacts["kmeans"].cluster_centers_.shape)

print("\nSaved regime names:")
print(regime_artifacts["regime_names"])

Saved regime detector input features:
['daily_return', 'ma20_ratio', 'ma50_ratio', 'volatility', 'momentum', 'rsi', 'volume_change', 'volume_ma20_ratio', 'regime_Bearish', 'regime_Bullish', 'regime_Sideways']

Saved scaler expects this many features:
2

Saved KMeans cluster centers shape:
(3, 2)

Saved regime names:
{1: 'Bearish', 2: 'Sideways', 0: 'Bullish'}


In [ ]:
scaler = regime_artifacts["regime_scaler"]
kmeans = regime_artifacts["kmeans"]

print("SCALER INPUT FEATURE NAMES:")
print(getattr(scaler, "feature_names_in_", "Names not saved"))

print("\nSCALER INPUT COUNT:")
print(scaler.n_features_in_)

print("\nKMEANS CLUSTER CENTERS:")
print(kmeans.cluster_centers_)

print("\nKMEANS INPUT FEATURE NAMES:")
print(getattr(kmeans, "feature_names_in_", "Names not saved"))

SCALER INPUT FEATURE NAMES:
['daily_return' 'volatility']

SCALER INPUT COUNT:
2

KMEANS CLUSTER CENTERS:
[[ 0.42842132 -0.23413269]
 [-0.84657952 -0.17220634]
 [ 0.34542158  3.62040429]]

KMEANS INPUT FEATURE NAMES:
Names not saved


In [3]:
print("X_test exists:", "X_test" in globals())
print("y_test exists:", "y_test" in globals())
print("automl exists:", "automl" in globals())

X_test exists: False
y_test exists: False
automl exists: False
